# AlignLLM — SFT + DPO Training (RunPod)

Fine-tune Llama-3.1-8B with QLoRA, then align with DPO.

**GPU**: RTX A5000 (24GB) | **Expected time**: ~90-120 min | **Cost**: ~$0.50

## 1. Setup

In [ ]:
!git clone https://github.com/SantoshAdabala/distill-align-llm.git
%cd distill-align-llm

In [ ]:
!pip install -q transformers accelerate peft datasets bitsandbytes trl
!pip install -e . -q

In [ ]:
# Check GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# Login to HuggingFace (Llama-3.1-8B is gated)
from huggingface_hub import login
login(token="YOUR_HF_TOKEN")  # Replace with your token

## 2. Load Model with QLoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "meta-llama/Llama-3.1-8B"

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Prepare for QLoRA training
model = prepare_model_for_kbit_training(model)

# LoRA config
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 3. SFT Training (~14 min)

In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

# Load instruction dataset
sft_dataset = load_dataset("tatsu-lab/alpaca", split="train[:5000]")
print(f"SFT dataset: {len(sft_dataset)} samples")

# Format for chat
def format_alpaca(example):
    if example.get("input", "").strip():
        prompt = f"{example['instruction']}\n\nInput: {example['input']}"
    else:
        prompt = example["instruction"]
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": example["output"]},
    ]
    return {"messages": messages}

sft_dataset = sft_dataset.map(format_alpaca, remove_columns=sft_dataset.column_names)
print(f"Formatted. Sample: {sft_dataset[0]['messages'][0]['content'][:100]}...")

In [ ]:
import time

sft_args = SFTConfig(
    output_dir="./outputs/sft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    warmup_steps=50,
    max_seq_length=1024,
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=10,
    save_steps=200,
    save_strategy="steps",
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
)

start = time.time()
sft_result = sft_trainer.train()
sft_time = (time.time() - start) / 60

print(f"\n{'='*60}")
print(f"SFT COMPLETE")
print(f"  Loss: {sft_result.training_loss:.4f}")
print(f"  Steps: {sft_result.global_step}")
print(f"  Time: {sft_time:.1f} min")
print(f"{'='*60}")

In [ ]:
# Save SFT adapter
SFT_ADAPTER_PATH = "./outputs/sft/final_adapter"
sft_trainer.save_model(SFT_ADAPTER_PATH)
print(f"SFT adapter saved to: {SFT_ADAPTER_PATH}")

## 4. DPO Training (~60-90 min)

**Improvements over v1:**
- Cleaner dataset (argilla/ultrafeedback-binarized-preferences-cleaned)
- Lower learning rate (1e-5 vs 5e-5)
- Proper validation evaluation (500 eval samples, eval every 100 steps)

In [ ]:
# Load cleaned preference dataset
pref_dataset = load_dataset(
    "argilla/ultrafeedback-binarized-preferences-cleaned",
    split="train",
)
print(f"Full preference dataset: {len(pref_dataset)} pairs")
print(f"Columns: {pref_dataset.column_names}")

# Split into train + eval
NUM_TRAIN = 5000
NUM_EVAL = 500

train_pref = pref_dataset.select(range(NUM_TRAIN))
eval_pref = pref_dataset.select(range(NUM_TRAIN, NUM_TRAIN + NUM_EVAL))
print(f"Train: {len(train_pref)}, Eval: {len(eval_pref)}")

In [ ]:
# Format for DPO (prompt, chosen, rejected)
def format_dpo(example):
    chosen_msgs = example.get("chosen", [])
    rejected_msgs = example.get("rejected", [])

    if isinstance(chosen_msgs, list) and len(chosen_msgs) >= 2:
        prompt = chosen_msgs[0]["content"] if isinstance(chosen_msgs[0], dict) else str(chosen_msgs[0])
        chosen = chosen_msgs[1]["content"] if isinstance(chosen_msgs[1], dict) else str(chosen_msgs[1])
    else:
        prompt = str(example.get("prompt", ""))
        chosen = str(chosen_msgs)

    if isinstance(rejected_msgs, list) and len(rejected_msgs) >= 2:
        rejected = rejected_msgs[1]["content"] if isinstance(rejected_msgs[1], dict) else str(rejected_msgs[1])
    else:
        rejected = str(rejected_msgs)

    return {"prompt": prompt, "chosen": chosen, "rejected": rejected}

train_pref = train_pref.map(format_dpo, remove_columns=train_pref.column_names)
eval_pref = eval_pref.map(format_dpo, remove_columns=eval_pref.column_names)

# Filter empty entries
def is_valid(ex):
    return len(ex["prompt"].strip()) > 0 and len(ex["chosen"].strip()) > 0 and len(ex["rejected"].strip()) > 0

train_pref = train_pref.filter(is_valid)
eval_pref = eval_pref.filter(is_valid)
print(f"After filtering — Train: {len(train_pref)}, Eval: {len(eval_pref)}")
print(f"Sample prompt: {train_pref[0]['prompt'][:100]}...")

In [ ]:
from trl import DPOConfig, DPOTrainer

# Reload model with SFT adapter for DPO
# (DPOTrainer needs a fresh model reference)
del sft_trainer
torch.cuda.empty_cache()

# Load SFT adapter
model.load_adapter(SFT_ADAPTER_PATH, adapter_name="sft")
model.set_adapter("sft")
print("SFT adapter loaded for DPO")

dpo_args = DPOConfig(
    output_dir="./outputs/dpo",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,           # Lower than v1 (was 5e-5)
    num_train_epochs=1,
    beta=0.1,
    max_length=1024,
    max_prompt_length=512,
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=20,
    eval_steps=100,               # Validate every 100 steps
    eval_strategy="steps",
    save_steps=250,
    save_strategy="steps",
    report_to="none",
)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,               # TRL creates frozen copy automatically
    args=dpo_args,
    train_dataset=train_pref,
    eval_dataset=eval_pref,
    processing_class=tokenizer,
)

start = time.time()
dpo_result = dpo_trainer.train()
dpo_time = (time.time() - start) / 60

print(f"\n{'='*60}")
print(f"DPO COMPLETE")
print(f"  Loss: {dpo_result.training_loss:.4f}")
print(f"  Steps: {dpo_result.global_step}")
print(f"  Time: {dpo_time:.1f} min")
print(f"{'='*60}")

In [ ]:
# Evaluate on held-out set
eval_metrics = dpo_trainer.evaluate()
print(f"\n{'='*60}")
print("DPO EVALUATION METRICS:")
for key, value in sorted(eval_metrics.items()):
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")
    else:
        print(f"  {key}: {value}")
print(f"{'='*60}")

# Key metrics to look for:
reward_acc = eval_metrics.get("eval_rewards/accuracies", 0)
reward_margin = eval_metrics.get("eval_rewards/margins", 0)
print(f"\n🎯 Reward Accuracy: {reward_acc:.2%}")
print(f"📊 Reward Margin: {reward_margin:.4f}")

if reward_acc >= 0.65:
    print("✅ Strong alignment!")
elif reward_acc >= 0.60:
    print("✅ Good alignment")
elif reward_acc >= 0.55:
    print("⚠️ Moderate — consider more data or lower LR")
else:
    print("❌ Weak — needs better data or hyperparameter tuning")

In [ ]:
# Save DPO adapter
DPO_ADAPTER_PATH = "./outputs/dpo/dpo_adapter"
dpo_trainer.save_model(DPO_ADAPTER_PATH)
print(f"DPO adapter saved to: {DPO_ADAPTER_PATH}")

## 5. Compare Responses: Base vs SFT vs DPO

In [ ]:
from peft import PeftModel
import gc

# Free memory
del dpo_trainer
del model
torch.cuda.empty_cache()
gc.collect()

PROMPTS = [
    "Explain gradient checkpointing in simple terms.",
    "How can I reduce GPU training costs?",
    "What is the difference between DPO and RLHF?",
    "Write a Python function to compute cosine similarity between two vectors.",
]

def generate(model, tokenizer, prompt, max_new_tokens=200):
    messages = [{"role": "user", "content": prompt}]
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        text = f"User: {prompt}\nAssistant:"
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, temperature=0.7, top_p=0.9, do_sample=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
# Load base model
print("Loading Base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.bfloat16
)

print("\n" + "="*80)
print("BASE MODEL RESPONSES")
print("="*80)
base_responses = []
for p in PROMPTS:
    r = generate(base_model, tokenizer, p)
    base_responses.append(r)
    print(f"\nPrompt: {p}")
    print(f"Response: {r[:300]}")
    print("-"*40)

In [ ]:
# Load SFT model
print("Loading SFT model...")
sft_model = PeftModel.from_pretrained(base_model, SFT_ADAPTER_PATH)

print("\n" + "="*80)
print("SFT MODEL RESPONSES")
print("="*80)
sft_responses = []
for p in PROMPTS:
    r = generate(sft_model, tokenizer, p)
    sft_responses.append(r)
    print(f"\nPrompt: {p}")
    print(f"Response: {r[:300]}")
    print("-"*40)

del sft_model
torch.cuda.empty_cache()

In [ ]:
# Load DPO model
print("Loading DPO model...")
dpo_model = PeftModel.from_pretrained(base_model, DPO_ADAPTER_PATH)

print("\n" + "="*80)
print("DPO MODEL RESPONSES")
print("="*80)
dpo_responses = []
for p in PROMPTS:
    r = generate(dpo_model, tokenizer, p)
    dpo_responses.append(r)
    print(f"\nPrompt: {p}")
    print(f"Response: {r[:300]}")
    print("-"*40)

del dpo_model, base_model
torch.cuda.empty_cache()

In [ ]:
# Side-by-side comparison
print("\n" + "="*80)
print("SIDE-BY-SIDE COMPARISON")
print("="*80)

for i, prompt in enumerate(PROMPTS):
    print(f"\n{'─'*80}")
    print(f"PROMPT: {prompt}")
    print(f"{'─'*80}")
    print(f"\n  [BASE]: {base_responses[i][:200]}")
    print(f"\n  [SFT]:  {sft_responses[i][:200]}")
    print(f"\n  [DPO]:  {dpo_responses[i][:200]}")

## 6. Save Results

In [ ]:
import json

results = {
    "sft_loss": sft_result.training_loss,
    "sft_steps": sft_result.global_step,
    "sft_time_min": round(sft_time, 1),
    "dpo_loss": dpo_result.training_loss,
    "dpo_steps": dpo_result.global_step,
    "dpo_time_min": round(dpo_time, 1),
    "dpo_lr": "1e-5",
    "dpo_dataset": "argilla/ultrafeedback-binarized-preferences-cleaned",
    "eval_reward_accuracy": eval_metrics.get("eval_rewards/accuracies", None),
    "eval_reward_margin": eval_metrics.get("eval_rewards/margins", None),
    "eval_loss": eval_metrics.get("eval_loss", None),
    "model": MODEL_ID,
    "gpu": torch.cuda.get_device_name(0),
    "comparisons": [
        {"prompt": p, "base": b, "sft": s, "dpo": d}
        for p, b, s, d in zip(PROMPTS, base_responses, sft_responses, dpo_responses)
    ],
}

with open("./outputs/v2_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Results saved to ./outputs/v2_results.json")
print("\nCopy this file to your local machine for the dashboard:")
print("  scp runpod:distill-align-llm/outputs/v2_results.json dashboard/v2_results.json")

In [ ]:
# Print final summary
print("\n" + "="*60)
print("TRAINING COMPLETE — FINAL SUMMARY")
print("="*60)
print(f"  SFT:  loss={sft_result.training_loss:.4f}, steps={sft_result.global_step}, time={sft_time:.1f}min")
print(f"  DPO:  loss={dpo_result.training_loss:.4f}, steps={dpo_result.global_step}, time={dpo_time:.1f}min")
print(f"  Eval: reward_acc={reward_acc:.2%}, margin={reward_margin:.4f}")
print(f"  Total time: {sft_time + dpo_time:.1f} min")
print("="*60)
print("\n⚠️  STOP YOUR RUNPOD POD NOW TO STOP BILLING!")